# Rb 87 D2: Cycling Transition, σ⁺ Optical Pumping

The D2 line of $^{87}$Rb connects the $5S_{1/2}$ ground state to the
$5P_{3/2}$ excited state at 780 nm. With circularly polarised ($\sigma^+$,
$q=+1$) light tuned to the $F=2 \to F'=3$ transition, the atoms are
optically pumped into the $m_F=+2$ **cycling state** — the sub-transition
$|F=2,\,m_F=+2\rangle \to |F'=3,\,m_{F'}=+3\rangle$ from which the only
allowed decay is back to $|F=2,\,m_F=+2\rangle$.

## Level structure

```
5P₃/₂  F'=3  mF = -3 -2 -1  0 +1 +2 +3
               ─────────────────────────
                    σ⁺ (q=+1) drive
5S₁/₂  F=2   mF = -2 -1  0 +1 +2
               ─────────────────
```

The $F=2 \to F'=3$ transition is **closed** under $\sigma^+$: spontaneous
decay from any $m_{F'}$ can only reach $m_F \leq m_{F'}$, so population
cascades toward $m_F=+2$ over many scattering cycles.

## Hyperfine structure construction

The `hyperfine.Atom1e` helper computes the Clebsch-Gordan-weighted coupling
and decay factors for the full $m_F$ sublevel structure. This eliminates
manual look-up of angular-momentum coefficients.

In [ ]:
import numpy as np
from maxwellbloch import hyperfine, mb_solve, plot

## Build the hyperfine structure

In [ ]:
# D2 line: 5S₁/₂ (J=1/2) → 5P₃/₂ (J=3/2), I = 3/2 for ⁸⁷Rb
Rb87_5s12_F2 = hyperfine.LevelF(I=1.5, J=0.5, F=2)
Rb87_5p32_F3 = hyperfine.LevelF(I=1.5, J=1.5, F=3)

atom1e = hyperfine.Atom1e(element="Rb", isotope="87")
atom1e.add_F_level(Rb87_5s12_F2)   # index 0: F=2, 5 mF sublevels
atom1e.add_F_level(Rb87_5p32_F3)   # index 1: F'=3, 7 mF sublevels

NUM_STATES = atom1e.get_num_mF_levels()
print(f"Total mF sublevels: {NUM_STATES}")  # 5 + 7 = 12

In [ ]:
# σ⁺ field: couples F=2 → F'=3 with q=+1 (Δm_F = +1)
q = 1
FIELD_CHANNELS = atom1e.get_coupled_levels(F_level_idxs_a=(0,), F_level_idxs_b=(1,))
FIELD_FACTORS  = atom1e.get_clebsch_hf_factors(F_level_idxs_a=(0,), F_level_idxs_b=(1,), q=q)
DECAY_CHANNELS = atom1e.get_coupled_levels(F_level_idxs_a=(0,), F_level_idxs_b=(1,))
DECAY_FACTORS  = atom1e.get_decay_factors(F_level_idxs_a=(0,), F_level_idxs_b=(1,))
ENERGIES       = atom1e.get_energies()

# Initial state: uniform across F=2 (mF = -2, -1, 0, +1, +2)
INITIAL_STATE = [1.0/5.0]*5 + [0.0]*7

# Detuning: on resonance with F=2 → F'=3
DETUNING = 0.0

print(f"Field channels: {len(FIELD_CHANNELS)}")
print(f"sum(|d_i|²) field factors: {np.sum(FIELD_FACTORS**2):.4f}")
print(f"sum(|d_i|²) decay factors: {np.sum(DECAY_FACTORS**2):.4f}")

## Solve: weak σ⁺ probe, optical pumping

In [ ]:
mb_solve_json = """
{{
  "atom": {{
    "decays": [
      {{
        "channels": {decay_channels},
        "rate": 1.0,
        "factors": {decay_factors}
      }}
    ],
    "energies": {energies},
    "fields": [
      {{
        "coupled_levels": {field_channels},
        "factors": {field_factors},
        "detuning": {detuning},
        "detuning_positive": true,
        "label": "probe",
        "rabi_freq": 1e-3,
        "rabi_freq_t_args": {{
          "ampl": 1.0,
          "centre": 0.0,
          "fwhm": 1.0
        }},
        "rabi_freq_t_func": "gaussian"
      }}
    ],
    "num_states": {num_states},
    "initial_state": {initial_state}
  }},
  "t_min": -2.0,
  "t_max": 10.0,
  "t_steps": 100,
  "z_min": -0.2,
  "z_max": 1.2,
  "z_steps": 100,
  "interaction_strengths": [1.0e2],
  "savefile": "mbs-Rb87_5s12_5p32_F23_q1-weak-pulse-decay"
}}
""".format(
    num_states=NUM_STATES,
    energies=ENERGIES,
    initial_state=INITIAL_STATE,
    detuning=DETUNING,
    field_channels=FIELD_CHANNELS,
    field_factors=FIELD_FACTORS.tolist(),
    decay_channels=DECAY_CHANNELS,
    decay_factors=DECAY_FACTORS.tolist(),
)

mbs = mb_solve.MBSolve().from_json_str(mb_solve_json)
mbs.mbsolve(recalc=False)
print("Done")

## Probe propagation

The space-time plot shows the weak probe pulse propagating through the
Rb vapour. On resonance, the pulse is absorbed as it passes through the
dense medium.

In [ ]:
fig = plot.field_spacetime(mbs, field_idx=0)
fig.update_layout(title="Probe |Ω(z, t)| — Rb 87 D2, F=2→F'=3, σ⁺")
fig.show(renderer='notebook_connected')

## Optical pumping: mF sublevel populations

The plot below shows the 12 sublevel populations at the input face as a
function of time. States 0–4 are the $F=2$ ground sublevels
($m_F = -2, -1, 0, +1, +2$) and states 5–11 are the $F'=3$ excited
sublevels ($m_{F'} = -3, \ldots, +3$).

Starting from a uniform distribution across $F=2$, $\sigma^+$ excitation
preferentially drives $\Delta m_F = +1$ transitions. After several
absorption/emission cycles, population accumulates in $m_F = +2$ — the
cycling state.

In [ ]:
fig = plot.population(mbs, state_indices=list(range(NUM_STATES)), z_idx=0)
fig.update_layout(title="mF sublevel populations at z = z_min — D2 σ⁺ optical pumping")
fig.show(renderer='notebook_connected')

## Ground-state populations only

Zooming in on just the $F=2$ ground sublevels makes the optical pumping
dynamics clearer: $m_F = +2$ grows at the expense of lower $m_F$ states.

In [ ]:
fig = plot.population(mbs, state_indices=list(range(5)), z_idx=0)
fig.update_layout(
    title="F=2 ground sublevel populations at z = z_min — D2 σ⁺ optical pumping",
)
fig.show(renderer='notebook_connected')

## Summary

- The $F=2 \to F'=3$ transition on the Rb 87 D2 line is a **closed cycling
  transition** under $\sigma^+$ excitation: spontaneous emission from
  $F'=3$ can only reach $F=2$, not $F=1$.
- Starting from uniform $m_F$ distribution, $\sigma^+$ optical pumping
  drives population toward $m_F=+2$ over multiple scattering cycles.
- The $|m_F=+2\rangle \to |m_{F'}=+3\rangle$ sub-transition is the
  **true cycling transition**: $m_{F'}=+3$ can only decay to $m_F=+2$
  (the only allowed $\Delta m_F \in \{-1, 0, +1\}$ channel from $m_{F'}=+3$
  to $F=2$), making atoms permanently trapped in the cycling loop.
- The `hyperfine.Atom1e` module constructs the full $m_F$ sublevel
  structure with correct Clebsch-Gordan-weighted coupling and decay
  factors, valid for any alkali $nS_{1/2} \to nP_J$ transition.

## References

1. D. A. Steck, *Rubidium 87 D Line Data* (2001, revision 2.2.2, 2021).
   Comprehensive Rb 87 spectroscopic data including D2 hyperfine constants,
   transition dipole elements, and natural linewidth. Available at
   https://steck.us/alkalidata.
2. C. J. Foot, *Atomic Physics*, Oxford University Press (2004).
   Chapter 7: laser cooling and optical pumping. Cycling transitions and
   the MOT.
3. H. J. Metcalf and P. van der Straten, *Laser Cooling and Trapping*,
   Springer (1999). Optical pumping and rate equations, Chapter 4.